In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil
import pickle

In [2]:

project = 'SurvSurfBenchmark_HousePrice'
ds_name = 'house_price_traj'
ds_df_dir = '/home/yc366/repos/survsurf_benchmark/dataset_split'
g_resol = 0.01
t_resol = 1
model_prefix = 'Joint'
col_traj_id = 'traj_id'

DIR_MODEL_SAVE = os.path.join('sksurv_models/more_g', project, model_prefix)

if os.path.exists(DIR_MODEL_SAVE):
    shutil.rmtree(DIR_MODEL_SAVE)
    os.mkdir(DIR_MODEL_SAVE)
else:
    os.mkdir(DIR_MODEL_SAVE)

In [3]:
from dataset_PropertyPrice import DatasetHousePrice

ds_train = DatasetHousePrice(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    t_resol=t_resol,
    split='train', 
    mode='first_cross_obs_only', 
    separate_g_from_feats=False
)
ds_val = DatasetHousePrice(
    df_dir=ds_df_dir, 
    ds_name=ds_name, 
    g_resol=g_resol, 
    t_resol=t_resol,
    split='val', 
    mode='first_cross_obs_only', 
    separate_g_from_feats=False
)


In [4]:
df_Xy_train = ds_train._get_df_Xy_trans_obs_more_g()
assert all(
    df_Xy_train.groupby(ds_train.colname_g)[col_traj_id].apply(lambda x: x.value_counts().max() == 1)
)
df_Xy_train['event_observed'] = df_Xy_train['event_observed'].astype(bool)
df_Xy_train = df_Xy_train.sort_values(col_traj_id)

df_Xy_val = ds_val._get_df_Xy_trans_obs()
assert all(
    df_Xy_val.groupby(ds_train.colname_g)[col_traj_id].apply(lambda x: x.value_counts().max() == 1)
)
df_Xy_val = df_Xy_val.sort_values(col_traj_id)
df_Xy_val['event_observed'] = df_Xy_val['event_observed'].astype(bool)

df_Xy_val_grid = ds_val._get_df_Xy_true_prob()
df_Xy_val_grid = df_Xy_val_grid.sort_values([col_traj_id,ds_train.colname_g])
df_Xy_val_grid['event_observed'] = df_Xy_val_grid['event_observed'].astype(bool)


In [5]:
df_Xy_train.head()

,traj_id,event_observed,duration,g_max_by_time,feat__is_new_build,feat__long,feat__lat,feat__frac_properties_n_beds__1,feat__frac_properties_n_beds__2,feat__frac_properties_n_beds__3,...,feat__deprived_4_dim,feat__soc_grade_AB,feat__soc_grade_C1,feat__soc_grade_C2,feat__soc_grade_DE,feat__prop_type_Detached,feat__prop_type_SemiD,feat__prop_type_Terraced,weight,is_t_trans
0,E06000001DetachedExisting,True,1.0,0.000580,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1
60,E06000001DetachedExisting,False,7.0,0.779746,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1
59,E06000001DetachedExisting,False,7.0,0.769746,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1
58,E06000001DetachedExisting,False,7.0,0.759746,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1
57,E06000001DetachedExisting,False,7.0,0.749746,0.0,0.021889,2.09093,-0.553782,-0.876423,-0.709005,...,0.088157,-1.329559,-1.741656,0.500371,1.905991,1.0,0.0,0.0,1,1


In [6]:
cols_y = ['event_observed', 'duration']
cols_X = np.r_[[ds_train.colname_g], df_Xy_train.columns[df_Xy_train.columns.str.startswith('feat')]]

In [7]:
cols_y

['event_observed', 'duration']

In [8]:
cols_X

array(['g_max_by_time', 'feat__is_new_build', 'feat__long', 'feat__lat',
       'feat__frac_properties_n_beds__1',
       'feat__frac_properties_n_beds__2',
       'feat__frac_properties_n_beds__3',
       'feat__frac_properties_n_beds__4',
       'feat__frac_properties_n_beds__>=5',
       'feat__tenure__owned_outright',
       'feat__tenure__owned_with_mortgage_or_shared_ownership',
       'feat__tenure__rented_from_local_auth',
       'feat__tenure__other_social_rent',
       'feat__tenure__private_rent_commercial',
       'feat__tenure__private_rent_other', 'feat__econ_act_part_time',
       'feat__econ_act_full_time', 'feat__econ_act_self_emp',
       'feat__econ_act_unemp', 'feat__econ_inact_ft_student',
       'feat__econ_inact_retired', 'feat__econ_inact_student',
       'feat__econ_inact_family_care',
       'feat__econ_inact_long_term_health', 'feat__econ_inact_other',
       'feat__econ_act_unemp_16to24', 'feat__econ_act_unemp_50to74',
       'feat__econ_act_never_worked', '

In [9]:
from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis, CoxPHSurvivalAnalysis
def get_non_cox_model(model_class):
    def get_model(n_jobs, random_state):
        try:
            return model_class(n_jobs=n_jobs, random_state=random_state, min_samples_leaf=3, n_estimators=1000)
        except TypeError:
            return model_class(random_state=random_state, min_samples_leaf=10, n_estimators=50)
    return get_model

def get_cox_model(model_class):
    def get_model(n_jobs, random_state):
        try:
            return model_class(fit_baseline_model=True, n_jobs=n_jobs)
        except TypeError:
            try:
                return model_class(fit_baseline_model=True)
            except TypeError:
                return model_class()
    return get_model

model_name_to_class = {
    f'{model_prefix}_{GradientBoostingSurvivalAnalysis.__name__}':get_non_cox_model(GradientBoostingSurvivalAnalysis),
    f'{model_prefix}_{RandomSurvivalForest.__name__}':get_non_cox_model(RandomSurvivalForest),
}

In [10]:
df_metrics = []
surv_funcs = dict()
event_to_model = dict()
seeds = [10,20,30,40,50]
for model_name, model_class in model_name_to_class.items():
    for seed in seeds:
        model_id = f'{model_name}_more_g_seed_{seed}'
        model = model_class(n_jobs=-1, random_state=seed)
        y = df_Xy_train[cols_y].to_records(index=False)
        X = df_Xy_train[cols_X]
        model.fit(X, y)

        model_path = os.path.join(DIR_MODEL_SAVE,f'{model_id}.pickle')
        with open(model_path, 'wb') as f:
            # Pickle the 'data' dictionary using the highest protocol available.
            pickle.dump(model, f)

        print(f'finished running model {model_name}')


finished running model Joint_GradientBoostingSurvivalAnalysis
finished running model Joint_GradientBoostingSurvivalAnalysis
finished running model Joint_GradientBoostingSurvivalAnalysis
finished running model Joint_GradientBoostingSurvivalAnalysis
finished running model Joint_GradientBoostingSurvivalAnalysis
finished running model Joint_RandomSurvivalForest
finished running model Joint_RandomSurvivalForest
finished running model Joint_RandomSurvivalForest
finished running model Joint_RandomSurvivalForest
finished running model Joint_RandomSurvivalForest
